In [1]:
# Step 4, cell 1: download the newest CFPB archive file (August 2026) and see what's inside
!wget -q -O aug_2026.zip "https://files.consumerfinance.gov/f/documents/CCDB_Export_21_August_2026.zip"
!ls -lh aug_2026.zip
!unzip -l aug_2026.zip

-rw-r--r-- 1 root root 11M Sep 14 14:42 aug_2026.zip
Archive:  aug_2026.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
177212547  2026-09-14 01:06   CCDB_Export_21_August_2026.csv
---------                     -------
177212547                     1 file


In [2]:
# Step 4, cell 2: open the data file inside the ZIP (everything read as text, like Project 1)
import zipfile
import pandas as pd

z = zipfile.ZipFile("aug_2026.zip")
names = z.namelist()
print("Files inside the ZIP:", names)

data_file = [n for n in names if n.lower().endswith((".csv", ".json"))][0]
print("Reading:", data_file)

if data_file.lower().endswith(".csv"):
    df = pd.read_csv(z.open(data_file), dtype=str)
else:
    df = pd.read_json(z.open(data_file), dtype=str)

print("Rows, columns:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Files inside the ZIP: ['CCDB_Export_21_August_2026.csv']
Reading: CCDB_Export_21_August_2026.csv
Rows, columns: (667140, 16)
Columns: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2026-08-04,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,MS,391XX,NaN,Web,2026-08-14,Closed with explanation,Yes,24937474
1,2026-08-01,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NC,27406,NaN,Web,2026-08-14,Closed with explanation,Yes,24850893
2,2026-08-02,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,MS,39211,NaN,Web,2026-08-14,Closed with explanation,Yes,24866553


In [3]:
# Step 4, cell 3: quick first look (date range, written stories, products, size in memory)
date_col = [c for c in df.columns if "received" in c.lower()][0]
narr_col = [c for c in df.columns if "narrative" in c.lower()][0]
prod_col = [c for c in df.columns if c.strip().lower() == "product"][0]

dates = pd.to_datetime(df[date_col], errors="coerce")
has_story = df[narr_col].fillna("").str.strip().ne("")

print("Date range:", dates.min().date(), "to", dates.max().date())
print("Dates that failed to read:", dates.isna().sum())
print(f"Complaints with a written story: {has_story.sum():,} of {len(df):,} ({has_story.mean():.1%})")
print("Memory used (MB):", round(df.memory_usage(deep=True).sum() / 1e6))
print()
print("Complaints by product:")
print(df[prod_col].value_counts().to_string())

Date range: 2026-08-01 to 2026-08-31
Dates that failed to read: 0
Complaints with a written story: 1 of 667,140 (0.0%)
Memory used (MB): 656

Complaints by product:
Product
Credit reporting or other personal consumer reports        619520
Debt collection                                             20398
Credit card                                                  7996
Checking or savings account                                  7552
Money transfer, virtual currency, or money service           3093
Mortgage                                                     2714
Vehicle loan or lease                                        2001
Payday loan, title loan, personal loan, or advance loan      1654
Student loan                                                 1407
Prepaid card                                                  491
Debt or credit management                                     314


In [4]:
# Step 4b: which months still have written stories? (checks every 2024-2026 archive file)
import os, zipfile, urllib.request
import pandas as pd

base = "https://files.consumerfinance.gov/f/documents/"
files = [
    "CCDB_Export_6_April_2024_through_July_2024.zip",
    "CCDB_Export_7_August_2024_through_October_2024.zip",
    "CCDB_Export_8_November_2024_through_December_2024.zip",
    "CCDB_Export_9_January_2025_through_February_2025.zip",
    "CCDB_Export_10_March_2025_through_April_2025.zip",
    "CCDB_Export_11_May_2025_through_June_2025.zip",
    "CCDB_Export_12_July_2025_through_August_2025.zip",
    "CCDB_Export_13_September_2025_through_October_2025.zip",
    "CCDB_Export_14_November_2025_through_December_2025.zip",
    "CCDB_Export_15_January_2026_through_February_2026.zip",
    "CCDB_Export_16_March_2026.zip",
    "CCDB_Export_17_April_2026.zip",
    "CCDB_Export_18_May_2026.zip",
    "CCDB_Export_19_June_2026.zip",
    "CCDB_Export_20_July_2026.zip",
    "CCDB_Export_21_August_2026.zip",
]
cols = ["Date received", "Product", "Consumer complaint narrative", "Complaint ID"]

pieces, sizes, ids_total, ids_seen = [], [], 0, set()
for f in files:
    urllib.request.urlretrieve(base + f, "tmp.zip")
    z = zipfile.ZipFile("tmp.zip")
    csv = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
    rows = 0
    for chunk in pd.read_csv(z.open(csv), dtype=str, usecols=cols, chunksize=500_000):
        rows += len(chunk)
        ids_total += len(chunk)
        ids_seen.update(chunk["Complaint ID"])
        month = pd.to_datetime(chunk["Date received"], errors="coerce").dt.to_period("M")
        story = chunk["Consumer complaint narrative"].fillna("").str.strip().ne("")
        bank = ~chunk["Product"].fillna("").str.startswith("Credit reporting")
        pieces.append(pd.DataFrame({"month": month, "story": story, "bank": bank, "bank_story": story & bank}))
    sizes.append((f, round(os.path.getsize("tmp.zip") / 1e6), rows))
    print(f"Done: {f}  ({rows:,} rows)")
    os.remove("tmp.zip")

all_rows = pd.concat(pieces)
by_month = all_rows.groupby("month").agg(
    complaints=("story", "size"),
    with_story=("story", "sum"),
    bank_complaints=("bank", "sum"),
    bank_with_story=("bank_story", "sum"),
)
by_month["story_share"] = (by_month["with_story"] / by_month["complaints"]).map("{:.1%}".format)

print()
print(f"Total rows: {ids_total:,}   Unique complaint IDs: {len(ids_seen):,}   Repeated IDs across files: {ids_total - len(ids_seen):,}")
print()
print(by_month.to_string())

Done: CCDB_Export_6_April_2024_through_July_2024.zip  (839,903 rows)
Done: CCDB_Export_7_August_2024_through_October_2024.zip  (824,514 rows)
Done: CCDB_Export_8_November_2024_through_December_2024.zip  (591,058 rows)
Done: CCDB_Export_9_January_2025_through_February_2025.zip  (811,019 rows)
Done: CCDB_Export_10_March_2025_through_April_2025.zip  (803,073 rows)
Done: CCDB_Export_11_May_2025_through_June_2025.zip  (827,343 rows)
Done: CCDB_Export_12_July_2025_through_August_2025.zip  (968,761 rows)
Done: CCDB_Export_13_September_2025_through_October_2025.zip  (1,023,636 rows)
Done: CCDB_Export_14_November_2025_through_December_2025.zip  (1,009,132 rows)
Done: CCDB_Export_15_January_2026_through_February_2026.zip  (1,038,383 rows)
Done: CCDB_Export_16_March_2026.zip  (614,945 rows)
Done: CCDB_Export_17_April_2026.zip  (618,293 rows)
Done: CCDB_Export_18_May_2026.zip  (619,940 rows)
Done: CCDB_Export_19_June_2026.zip  (620,794 rows)
Done: CCDB_Export_20_July_2026.zip  (672,036 rows)
Done: